# Week 5 — High-Dimensional Neural Data: From Features to Biological Structure

**Applied Inference II: dimensionality reduction, clustering, and interpreting large feature spaces**

Today we will move from analyzing one biological measurement at a time to analyzing datasets where each observation has many measurements.

Scientific question:

> When we measure hundreds or thousands of neural features, how can we identify meaningful biological structure without confusing mathematical patterns for biology?

We will work with a synthetic single-cell neuroscience dataset:
- each row represents a neuron;
- each column represents a measured feature;
- metadata describes biological conditions.

The main ideas:
1. high-dimensional data behaves differently than low-dimensional data;
2. scaling and preprocessing affect interpretation;
3. PCA creates new axes that summarize variation;
4. clustering finds mathematical groups, not automatically biological cell types.


In [ ]:
from io import BytesIO
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

GITHUB_RAW = "https://raw.githubusercontent.com/willi3by/Survey_of_Methods_in_Comp_Neuro_Fall_2026/main/Data/Week_05"

def github_csv(filename):
    url = f"{GITHUB_RAW}/{filename}"
    req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return pd.read_csv(BytesIO(urlopen(req).read()))

data = github_csv("neural_features.csv")
metadata = github_csv("neural_metadata.csv")

data.head()


## 1. From observations to feature spaces

In Week 4, each cell was represented by a small number of predictors:

- distance from a target site;
- treatment;
- morphology measurements.

Now each neuron may have dozens or hundreds of measured features.

The data matrix has the form:

\[
X_{n \times p}
\]

where:

- \(n\) = number of neurons (observations)
- \(p\) = number of measured features

A major challenge is that distance in high-dimensional space becomes harder to interpret.

Before using dimensionality reduction, inspect:
- number of observations;
- number of features;
- whether features have comparable scales.


In [ ]:
features = data.select_dtypes(include='number')

print("Observations:", features.shape[0])
print("Features:", features.shape[1])

features.describe().T.head()


## 2. Why scaling matters

Features may be measured on very different scales.

For example:
- fluorescence intensity may range from 0–5000;
- morphology measurements may range from 0–20;
- electrophysiology features may use different units.

PCA is variance-based. Without scaling, the largest numerical variables can dominate the analysis.

We standardize each feature:

\[
z = \frac{x - \mu}{\sigma}
\]

so each feature contributes more equally.


In [ ]:
X = StandardScaler().fit_transform(features)

pd.DataFrame(X, columns=features.columns).head()


## 3. Principal component analysis (PCA)

PCA rotates the feature space to create new axes:

\[
PC_1, PC_2, PC_3, ...
\]

Each principal component is a weighted combination of the original features.

The first components explain the largest amount of variation.

Important interpretation:

A PCA plot is a visualization of variation, not proof of biological categories.


In [ ]:
pca = PCA()
pcs = pca.fit_transform(X)

variance = pca.explained_variance_ratio_

plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(variance)+1), variance.cumsum(), marker='o')
plt.xlabel("Principal component")
plt.ylabel("Cumulative variance explained")
plt.title("PCA variance explained")
plt.show()


In [ ]:
pc_df = pd.DataFrame(
    pcs[:, :2],
    columns=["PC1", "PC2"]
).join(metadata)

plt.figure(figsize=(6,5))
for group, subset in pc_df.groupby("cell_type"):
    plt.scatter(subset.PC1, subset.PC2, label=group, alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Neurons projected into PCA space")
plt.legend()
plt.show()


### Interpretation prompt

Before looking at labels:

1. Do you see clusters in PCA space?
2. What does a point represent?
3. Does distance between points have a biological meaning?
4. What information was lost by reducing many dimensions to two?


## 4. What features drive the separation?

The PCA axes are combinations of the original features.

The loadings tell us which measurements contribute most to each component.

Large positive or negative loadings indicate features strongly associated with that component.


In [ ]:
loadings = pd.DataFrame(
    pca.components_[:2].T,
    columns=["PC1_loading", "PC2_loading"],
    index=features.columns
)

loadings.abs().sort_values(
    "PC1_loading",
    ascending=False
).head(10)


## 5. Clustering

Clustering asks:

> Can we group observations based on similarity in feature space?

Here we use k-means.

The algorithm:
1. chooses cluster centers;
2. assigns observations to the closest center;
3. updates centers;
4. repeats until stable.

Important:

A cluster is a mathematical grouping. It is not automatically a biological cell type.


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=0, n_init=20)

clusters = kmeans.fit_predict(pcs[:, :10])

cluster_df = pc_df.copy()
cluster_df["cluster"] = clusters

silhouette_score(pcs[:, :10], clusters)


In [ ]:
plt.figure(figsize=(6,5))

for cluster, subset in cluster_df.groupby("cluster"):
    plt.scatter(
        subset.PC1,
        subset.PC2,
        label=f"Cluster {cluster}",
        alpha=0.7
    )

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters in PCA space")
plt.legend()
plt.show()


## 6. Common failure modes

High-dimensional analyses can produce convincing-looking results that are misleading.

Potential problems:

### Technical variation
A cluster may represent:
- batch;
- sequencing depth;
- staining intensity;
- microscope differences.

### Feature selection
A few highly variable features can dominate the analysis.

### Cluster number
The number of clusters is a modeling choice, not a discovery made by the algorithm.

### Biological interpretation
Clusters require validation:
- additional experiments;
- known markers;
- independent datasets.


## Final discussion

Consider the following:

1. If PCA separates neurons by treatment, does that prove treatment caused the separation?
2. If clustering identifies three groups, does that mean there are exactly three cell types?
3. What additional evidence would make you more confident that a discovered group is biologically meaningful?

**Key idea:**
Dimensionality reduction and clustering are tools for exploring structure. They generate hypotheses; they do not replace biological validation.
